# 03 - Why GPUs Matter

**Goal:** Understand why deep learning needs GPUs and what makes them different from CPUs.

---

## The Core Problem

Neural networks are **matrix multiplications** repeated millions of times.

```
One layer: output = activation(input @ weights + bias)
```

A single forward pass through SwinUNETR (your production model) involves:
- ~62 million parameters
- Billions of floating-point operations
- For a 96x96x96 3D volume

Training requires doing this thousands of times, plus backpropagation.

## CPU vs GPU: The Difference

| | CPU | GPU |
|---|-----|-----|
| **Cores** | 8-16 powerful cores | 1000s of small cores |
| **Optimized for** | Sequential tasks, branching logic | Parallel math operations |
| **Analogy** | One professor solving problems | 1000 students doing simple math |

Matrix multiplication is **embarrassingly parallel** - each output element can be computed independently.

```
Matrix A (1000x1000) @ Matrix B (1000x1000) = C (1000x1000)

CPU: Compute each of 1,000,000 elements mostly sequentially
GPU: Compute thousands of elements simultaneously
```

In [3]:
import torch
import time

# Check what's available
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU only (that's fine for learning)")

PyTorch version: 2.6.0+cpu
CUDA available: False
Running on CPU only (that's fine for learning)


In [4]:
# Benchmark: Matrix multiplication CPU vs GPU
def benchmark_matmul(size, device, runs=10):
    """Time matrix multiplication on a device"""
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)
    
    # Warmup
    for _ in range(3):
        c = torch.matmul(a, b)
    
    if device == 'cuda':
        torch.cuda.synchronize()
    
    # Timed runs
    start = time.perf_counter()
    for _ in range(runs):
        c = torch.matmul(a, b)
    if device == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    
    return elapsed / runs

# Test different sizes
sizes = [512, 1024, 2048, 4096]
print("Matrix multiplication benchmark (seconds per operation):\n")
print(f"{'Size':>6} | {'CPU':>10} | {'GPU':>10} | {'Speedup':>8}")
print("-" * 45)

for size in sizes:
    cpu_time = benchmark_matmul(size, 'cpu')
    
    if torch.cuda.is_available():
        gpu_time = benchmark_matmul(size, 'cuda')
        speedup = cpu_time / gpu_time
        print(f"{size:>6} | {cpu_time:>10.4f} | {gpu_time:>10.4f} | {speedup:>7.1f}x")
    else:
        print(f"{size:>6} | {cpu_time:>10.4f} | {'N/A':>10} | {'N/A':>8}")

Matrix multiplication benchmark (seconds per operation):

  Size |        CPU |        GPU |  Speedup
---------------------------------------------
   512 |     0.0017 |        N/A |      N/A
  1024 |     0.0090 |        N/A |      N/A
  2048 |     0.0500 |        N/A |      N/A
  4096 |     0.3095 |        N/A |      N/A


## GPU Memory: The Bottleneck

GPUs are fast but have **limited memory** (typically 8-24 GB for consumer cards, 40-80 GB for datacenter).

During training, you need to store:
1. **Model weights** (~250 MB for SwinUNETR in float32)
2. **Input batch** (e.g., 4 images of 96x96x96 = ~14 MB)
3. **Activations** (intermediate outputs for backprop - this is the big one)
4. **Gradients** (same size as weights)
5. **Optimizer state** (Adam stores 2x weights)

This is why:
- Batch size is limited by GPU memory
- 3D medical imaging is memory-hungry (volumes, not 2D images)
- Your production code uses sliding window inference (process patches, not full volume)

In [5]:
# Memory calculation example
import numpy as np

def calculate_tensor_memory(shape, dtype='float32'):
    """Calculate memory in MB for a tensor"""
    bytes_per_element = {'float32': 4, 'float16': 2, 'int8': 1}[dtype]
    total_elements = np.prod(shape)
    return total_elements * bytes_per_element / (1024 * 1024)

# Your production model input
region_input = (1, 1, 96, 96, 96)  # batch, channels, depth, height, width
instance_input = (1, 1, 128, 128, 96)

print("Memory for single inputs:")
print(f"  Region model input (96x96x96): {calculate_tensor_memory(region_input):.2f} MB")
print(f"  Instance model input (128x128x96): {calculate_tensor_memory(instance_input):.2f} MB")

# Batch of 4
batch_region = (4, 1, 96, 96, 96)
print(f"\nBatch of 4 (region): {calculate_tensor_memory(batch_region):.2f} MB")

# Model weights (rough estimate for SwinUNETR)
params = 62_000_000  # 62M parameters
print(f"\nModel weights (~62M params): {params * 4 / (1024*1024):.2f} MB")
print(f"+ Gradients (same size): {params * 4 / (1024*1024):.2f} MB")
print(f"+ Adam optimizer state (2x): {params * 4 * 2 / (1024*1024):.2f} MB")

Memory for single inputs:
  Region model input (96x96x96): 3.38 MB
  Instance model input (128x128x96): 6.00 MB

Batch of 4 (region): 13.50 MB

Model weights (~62M params): 236.51 MB
+ Gradients (same size): 236.51 MB
+ Adam optimizer state (2x): 473.02 MB


## PyTorch Device Management

Moving data between CPU and GPU is explicit in PyTorch:

```python
# Create on CPU (default)
x = torch.randn(100, 100)

# Move to GPU
x = x.to('cuda')  # or x.cuda()

# Move back to CPU
x = x.to('cpu')   # or x.cpu()

# Create directly on GPU
x = torch.randn(100, 100, device='cuda')
```

**Important:** All tensors in an operation must be on the same device!

In [6]:
# Device management in practice

# Pattern 1: Check and use best available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create tensor on that device
x = torch.randn(1000, 1000, device=device)
print(f"Tensor device: {x.device}")

# This is the pattern you'll see in production code:
# model = model.to(device)
# input = input.to(device)
# output = model(input)

Using device: cpu
Tensor device: cpu


In [7]:
# What happens if tensors are on different devices?
if torch.cuda.is_available():
    a = torch.randn(10, device='cpu')
    b = torch.randn(10, device='cuda')
    
    try:
        c = a + b  # This will fail!
    except RuntimeError as e:
        print(f"Error: {e}")
        print("\nSolution: Move both to same device first")
        c = a.to('cuda') + b
        print(f"Result device: {c.device}")
else:
    print("GPU not available - this error demo requires CUDA")

GPU not available - this error demo requires CUDA


## Mixed Precision (float16)

Modern GPUs support **half-precision** (float16) which:
- Uses half the memory
- Is faster on modern GPUs (Tensor Cores)
- Slightly less precise (usually fine for deep learning)

Your production code uses this:
```python
with torch.amp.autocast('cuda'):  # Automatic Mixed Precision
    prediction = model(image)
```

PyTorch automatically chooses float16 for safe operations, float32 for sensitive ones.

In [8]:
# Memory comparison: float32 vs float16
shape = (4, 1, 128, 128, 96)  # Batch of 4 volumes

print(f"Shape: {shape}")
print(f"float32: {calculate_tensor_memory(shape, 'float32'):.2f} MB")
print(f"float16: {calculate_tensor_memory(shape, 'float16'):.2f} MB")
print(f"Savings: 50%")

Shape: (4, 1, 128, 128, 96)
float32: 24.00 MB
float16: 12.00 MB
Savings: 50%


## Connection to Your Production Code

From `segmentation/src/inference_engine_v016.py`:

```python
# Device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to GPU
model = model.to(device)

# Mixed precision inference
with torch.amp.autocast('cuda'):
    prediction = sliding_window_inference(
        input_image,
        roi_size=[96, 96, 96],  # Process in patches
        sw_batch_size=4,        # 4 patches at once
        ...
    )

# Clear GPU memory after heavy operations
torch.cuda.empty_cache()
```

The sliding window approach handles volumes larger than GPU memory by processing patches.

## Summary

| Concept | What it means |
|---------|---------------|
| **GPU** | Thousands of small cores for parallel math |
| **CUDA** | NVIDIA's GPU programming platform |
| **GPU Memory** | Limited (8-80 GB), main bottleneck |
| **Device** | Where tensors live (CPU or GPU) |
| **Mixed Precision** | Use float16 for speed/memory, float32 for accuracy |
| **Sliding Window** | Process large inputs in patches |

**Key takeaway:** GPUs make deep learning practical by parallelizing matrix math. The main constraint is memory, not compute.

**Next:** What is a CNN? How do convolutions work?